# Physical-quantity probes

Download the analysis data as described in the root README, or run `scripts/export_embeddings.py`.
Train probes on frozen training embeddings and report held-out evaluation R².
Linear: StandardScaler + Ridge (α=10⁻³). MLP: widths 256/128, with scaling fitted on train data
and early stopping using only an internal training split. Angles use sine/cosine targets;
grouped R² is the uniform average across target components. See `probes.py` for the implementations.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from plot_style import apply_matplotlib_style

apply_matplotlib_style()

from analysis.probes import load_split, physical_targets, fit_probe

EXPORT_ROOT = PLANNING / "outputs/embeddings"
METHODS = {"smwm": "SMWM", "sigreg": "SIGReg", "forward_only": "Fwd.-only"}
ENVS = {"tworoom": "TwoRoom", "reacher": "Reacher", "pusht": "Push-T", "ogbcube": "Cube"}
PROBES = ["linear", "mlp"]  # select ["linear"] for a quick check


In [ ]:
rows = []
for env, environment in ENVS.items():
    for method, label in METHODS.items():
        directory = EXPORT_ROOT / env / method
        if not (directory / "train_embeddings.pt").is_file() or not (directory / "eval_embeddings.pt").is_file():
            print(f"Missing {directory}; export this model's embeddings first.")
            continue
        z_train, train_arrays = load_split(directory, "train")
        z_eval, eval_arrays = load_split(directory, "eval")
        train_targets, eval_targets = physical_targets(env, train_arrays), physical_targets(env, eval_arrays)
        for quantity in train_targets:
            for probe in PROBES:
                print(f"{environment}/{label}: {quantity}, {probe}")
                score = fit_probe(z_train, train_targets[quantity], z_eval, eval_targets[quantity], probe)
                rows.append(dict(environment=environment, quantity=quantity, method=label, probe=probe, r2=score))
scores = pd.DataFrame(rows)
if not scores.empty:
    table = scores.pivot(index=["environment", "quantity"], columns=["probe", "method"], values="r2")
    order = [(probe, label) for probe in PROBES for label in METHODS.values()]
    table = table.reindex(columns=pd.MultiIndex.from_tuples(order)).reindex(ENVS.values(), level="environment")
    display(table.round(3))
    output = PLANNING / "outputs/analysis"
    output.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output / "physical_probes.csv", index=False)
    table.to_latex(output / "physical_probes.tex", float_format="%.3f", multicolumn=True, na_rep="—")
